# Figure S15 read-based model ablation

Sidecar experiment for the Liang inactive-vs-active classifier in `figure_s15.ipynb`.
**Does not** overwrite `phage_activity_model_full.joblib` or edit the manuscript notebook.

Protocol matches the figure notebook: `pr_cat == "TP"` = inactive, GroupKFold by `group`,
`random_state=42`, `class_weight="balanced"`. Primary metric is OOF recall at 95/90/85% precision.
Secondary: AUROC, AUPRC, MCC. Sanity slice: mixed-label species (TP in some libraries, FP in others).

Training table: `classifier_train_eval_features.tsv` (3,460 bulk TP/FP rows).

| Phase | Test |
|---|---|
| 1 | Reproduce 86-feature RF; drop duplicate/dead columns |
| 2 | Tight occupancy+quality core; occupancy / ANI / quality / lifestyle blocks |
| 3 | Tuned RF vs `HistGradientBoostingClassifier` vs isotonic calibration |
| 4 | Add `occ_struct`, `occ_int`, `log_depth_ratio` from bulk (`*_unenriched`) gene coverage |

Helper: `_extract/run_s15_model_ablation.py`. Results: `model_ablation/comparison.tsv`.


In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
FIG = cwd if (cwd / 'classifier_train_eval_features.tsv').exists() else cwd / 'uhvdb-manuscript' / 'figure_s15'
helper = FIG / '_extract' / 'run_s15_model_ablation.py'
if not helper.exists():
    raise FileNotFoundError(helper)

spec = importlib.util.spec_from_file_location('run_s15_model_ablation', helper)
A = importlib.util.module_from_spec(spec)
spec.loader.exec_module(A)
print('FIG', FIG)
print('helper', helper)


## Load training table

Same 86 features as `OPTIMAL_FEATURES` in `figure_s15.ipynb`. Mixed-label species are the
sanity check that occupancy (not genome identity) is doing the work.


In [ ]:
df, feature_cols, mixed_mask, mixed_species = A.load_training_table()
print('n mixed-label species', len(mixed_species))
print('hygiene drops present:', sum(c in feature_cols for c in A.DROP_HYGIENE), '/', len(A.DROP_HYGIENE))
print('tight core', len(A.TIGHT_CORE), A.TIGHT_CORE)


## Run all phases

This cell runs Phases 1–4 (RF hygiene, grouped ablation, HGB/calibration, module occupancy)
and writes `model_ablation/comparison.tsv` plus `model_ablation/decision.txt`.

Promotion into `figure_s15.ipynb` happens **only** if recall at 95% precision rises and
mixed-species AUROC does not drop. The joblib is left untouched either way.


In [ ]:
res_df, best, promote = A.main()
display(res_df.sort_values('recall_95', ascending=False))
print('best', best['config'], 'promote', promote)


## Comparison table and decision

Published kitchen-sink RF: AUROC 0.826 / AUPRC 0.922 / **41.8% recall at 95% precision**.
Promote only if a config beats reproduced recall@95% without harming mixed-species AUROC.


In [ ]:
out = FIG / 'model_ablation'
cmp = pd.read_csv(out / 'comparison.tsv', sep='\t')
display(cmp.sort_values('recall_95', ascending=False).reset_index(drop=True))
print((out / 'decision.txt').read_text())
png = out / 'recall_at_95.png'
if png.exists():
    display(Markdown(f'![recall@95]({png})'))
